# Multi-Task Live Demo

통합된 모델을 불러와 라인 트레이싱과 손가락 분류 모드를 실시간으로 스위칭합니다.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import cv2
import numpy as np
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Robot, Camera, bgr8_to_jpeg
import torch.nn.functional as F

class MultiTaskResNet(nn.Module):
    def __init__(self):
        super(MultiTaskResNet, self).__init__()
        resnet = models.resnet18(pretrained=False)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.xy_head = nn.Linear(512, 2)
        self.finger_head = nn.Linear(512, 6)

    def forward(self, x, mode='line'):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        if mode == 'line': return self.xy_head(x)
        elif mode == 'finger': return self.finger_head(x)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiTaskResNet()
try:
    model.load_state_dict(torch.load('best_model_multi.pth'))
    print("Model loaded successfully")
except:
    print("No model found. Please train first.")
model = model.to(device)
model.eval()

camera = Camera()
image_widget = widgets.Image(format='jpeg', width=224, height=224)
traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)
robot = Robot()

mode_select = widgets.ToggleButtons(options=['Line Tracing', 'Finger Classification'], description='Mode:')
output_label = widgets.HTML(value='')

def preprocess(camera_value):
    x = camera_value
    x = cv2.resize(x, (224, 224))
    x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB)
    x = x.transpose((2, 0, 1))
    x = torch.from_numpy(x).float()
    x = transforms.functional.normalize(x, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    x = x.to(device)
    x = x[None, ...]
    return x


In [ ]:
def update(change):
    new_image = change['new']
    x = preprocess(new_image)
    
    mode = mode_select.value
    with torch.no_grad():
        if mode == 'Line Tracing':
            y = model(x, mode='line')
            xy = y.detach().cpu().numpy().flatten()
            x_val, y_val = xy[0], xy[1]
            
            speed_slider = 0.2
            steering_slider = 0.05
            angle = np.arctan2(x_val, y_val)
            pid = angle * steering_slider
            
            robot.left_motor.value = max(min(speed_slider + pid, 1.0), 0.0)
            robot.right_motor.value = max(min(speed_slider - pid, 1.0), 0.0)
            output_label.value = f"<h4>Line Mode</h4>x={x_val:.3f}, y={y_val:.3f}"
            
        elif mode == 'Finger Classification':
            robot.stop() # Stop moving while classifying finger
            y = model(x, mode='finger')
            y = F.softmax(y, dim=1)
            prob, pred = torch.max(y, 1)
            output_label.value = f"<h4>Finger Mode</h4>Class: {pred.item()} (prob: {prob.item():.2f})"

update_camera = camera.observe(update, names='value')

display(widgets.VBox([mode_select, image_widget, output_label]))

In [ ]:
# Run this to stop the camera and robot
# camera.unobserve(update_camera, names='value')
# robot.stop()
# camera.stop()